# Echelon Chess Engine - Colab Training (C++ Optimized)

This notebook trains the Echelon chess engine using high-performance C++ backend for AlphaZero-style self-play.

**Before running:**
1. Go to Runtime > Change runtime type > GPU (T4)
2. Run all cells in order

In [ ]:
# Clone repository
!git lfs install
!git clone https://github.com/falloficarus22/echelon.git
%cd echelon
!git lfs pull

In [ ]:
!ls -lh checkpoints/*.pt

In [ ]:
# Install dependencies
!pip install -r requirements.txt -q
!pip install pybind11 -q

In [ ]:
# Remove any pre-compiled .so files (they won't work on Colab)
!rm -f cpp/*.so *.so
print("Cleaned up pre-compiled binaries")

In [ ]:
%%bash
set -e
echo "Compiling C++ backend using build_cpp.sh..."
./build_cpp.sh
echo
ls -lh cpp/echelon_cpp* || true
echo "\n✓ C++ backend compiled successfully!"

In [ ]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Verify C++ is loaded
import sys
sys.path.insert(0, './cpp')
import echelon_cpp
echelon_cpp.init()

# Import the fast worker
from selfplay_cpp import CPPSelfPlayWorker
print("✓ C++ self-play ready")

## Training

**Performance on T4 GPU:**
- Self-play: ~2-10 seconds per game (vs 30-60s in pure Python)
- Training: Limited by neural network, not chess engine
- Total speedup: ~60x faster than pure Python

The script will run 10 iterations with 5 games each.

In [ ]:
# Start training using C++ backend (small smoke settings)
!python train_cpp.py --iterations 10 --games_per_iter 5 --mcts_sims 200 --resume checkpoints/checkpoint_iter_76.pt

## Save Results to Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy checkpoints to Drive
import os

drive_path = '/content/drive/MyDrive/echelon_checkpoints'
os.makedirs(drive_path, exist_ok=True)

# Copy all .pt files
!cp *.pt {drive_path}/ 2>/dev/null || echo 'No checkpoints found yet'
print(f"Checkpoints saved to {drive_path}")